In [2]:
# Dependencies for this project are listed in requirements.txt
# Install with: pip install -r requirements.txt
print("Environment ready.")

Environment ready.


In [3]:
# ==========================================
# REQUIRED LIBRARIES IMPORT
# ==========================================

# Pandas: Main tool for data manipulation and analysis of tabular data
import pandas as pd

# NumPy: Numerical operations and array handling
import numpy as np

# SQLAlchemy: ORM and database toolkit for loading data into SQLite
from sqlalchemy import create_engine, text

# OS and pathlib: File system navigation and path management
import os
from pathlib import Path

# Display configuration for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Libraries successfully loaded and environment configured.")

Libraries successfully loaded and environment configured.


## STEP 1: EXTRACT

In [4]:
# ==========================================
# 1.1 LOAD ALL CSV FILES FROM RAW DATA FOLDER
# ==========================================

# Base path to raw data folder (relative to notebook location)
RAW_PATH = Path("data/raw")

# Dictionary mapping table names to their CSV filenames
csv_files = {
    "orders":       "olist_orders_dataset.csv",
    "order_items":  "olist_order_items_dataset.csv",
    "customers":    "olist_customers_dataset.csv",
    "products":     "olist_products_dataset.csv",
    "sellers":      "olist_sellers_dataset.csv",
    "payments":     "olist_order_payments_dataset.csv",
    "reviews":      "olist_order_reviews_dataset.csv",
    "geolocation":  "olist_geolocation_dataset.csv",
    "translations": "product_category_name_translation.csv"
}

# Load each CSV into a DataFrame and store in a dictionary
dataframes = {}

for name, filename in csv_files.items():
    filepath = RAW_PATH / filename
    dataframes[name] = pd.read_csv(filepath)
    print(f"  {name:<15} loaded   {len(dataframes[name]):>7,} rows   {len(dataframes[name].columns)} columns")

print("\nAll files loaded successfully.")

  orders          loaded    99,441 rows   8 columns
  order_items     loaded   112,650 rows   7 columns
  customers       loaded    99,441 rows   5 columns
  products        loaded    32,951 rows   9 columns
  sellers         loaded     3,095 rows   4 columns
  payments        loaded   103,886 rows   5 columns
  reviews         loaded    99,224 rows   7 columns
  geolocation     loaded   1,000,163 rows   5 columns
  translations    loaded        71 rows   2 columns

All files loaded successfully.


In [5]:
# ==========================================
# 1.2 INITIAL INSPECTION
# ==========================================

# Display shape, columns and first 2 rows for each loaded table
for name, df in dataframes.items():
    print(f"{'='*60}")
    print(f"TABLE: {name.upper()}   ({df.shape[0]:,} rows x {df.shape[1]} columns)")
    print(f"{'='*60}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nFirst 2 rows:")
    print(df.head(2).to_string())
    print()

TABLE: ORDERS   (99,441 rows x 8 columns)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

First 2 rows:
                           order_id                       customer_id order_status order_purchase_timestamp    order_approved_at order_delivered_carrier_date order_delivered_customer_date order_estimated_delivery_date
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15          2017-10-04 19:55:00           2017-10-10 21:25:13           2017-10-18 00:00:00
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27          2018-07-26 14:31:00           2018-08-07 15:27:45           2018-08-13 00:00:00

TABLE: ORDER_ITEMS   (112,650 rows x 7 columns)
Columns: ['order_id', 'order_item_id', '

## STEP 2: EXPLORE & VALIDATE

In [6]:
# ==========================================
# 2.1 SHAPE AND DATA TYPES
# ==========================================

# Display dtypes and memory usage for each table
for name, df in dataframes.items():
    print(f"{'='*60}")
    print(f"TABLE: {name.upper()}")
    print(f"{'='*60}")
    print(df.dtypes.to_string())
    print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print()

TABLE: ORDERS
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str

Memory usage: 58.97 MB

TABLE: ORDER_ITEMS
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64

Memory usage: 39.43 MB

TABLE: CUSTOMERS
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str

Memory usage: 29.62 MB

TABLE: PRODUCTS
product_id                        str
product_category_name             str
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty      

In [7]:
# ==========================================
# 2.2 NULL VALUES
# ==========================================

# Display null count and percentage per column for each table
for name, df in dataframes.items():
    null_counts = df.isnull().sum()
    null_pct = (null_counts / len(df) * 100).round(2)
    null_summary = pd.DataFrame({
        "null_count": null_counts,
        "null_pct": null_pct
    })
    null_summary = null_summary[null_summary["null_count"] > 0]

    print(f"{'='*60}")
    print(f"TABLE: {name.upper()}")
    print(f"{'='*60}")
    if null_summary.empty:
        print("No null values found.")
    else:
        print(null_summary.to_string())
    print()

TABLE: ORDERS
                               null_count  null_pct
order_approved_at                     160      0.16
order_delivered_carrier_date         1783      1.79
order_delivered_customer_date        2965      2.98

TABLE: ORDER_ITEMS
No null values found.

TABLE: CUSTOMERS
No null values found.

TABLE: PRODUCTS
                            null_count  null_pct
product_category_name              610      1.85
product_name_lenght                610      1.85
product_description_lenght         610      1.85
product_photos_qty                 610      1.85
product_weight_g                     2      0.01
product_length_cm                    2      0.01
product_height_cm                    2      0.01
product_width_cm                     2      0.01

TABLE: SELLERS
No null values found.

TABLE: PAYMENTS
No null values found.

TABLE: REVIEWS
                        null_count  null_pct
review_comment_title         87656     88.34
review_comment_message       58247     58.70

TABLE: GE

In [8]:
# ==========================================
# 2.3 RELATIONSHIP VALIDATION
# ==========================================

# Verify referential integrity between key tables
# Check that foreign keys in child tables exist in parent tables

checks = [
    ("order_items.order_id",   "order_items",  "order_id",   "orders",    "order_id"),
    ("order_items.product_id", "order_items",  "product_id", "products",  "product_id"),
    ("order_items.seller_id",  "order_items",  "seller_id",  "sellers",   "seller_id"),
    ("payments.order_id",      "payments",     "order_id",   "orders",    "order_id"),
    ("reviews.order_id",       "reviews",      "order_id",   "orders",    "order_id"),
    ("orders.customer_id",     "orders",       "customer_id","customers", "customer_id"),
]

print("Referential integrity checks:")
print(f"{'='*60}")

for label, child_name, child_col, parent_name, parent_col in checks:
    child_df  = dataframes[child_name]
    parent_df = dataframes[parent_name]

    child_keys  = set(child_df[child_col].unique())
    parent_keys = set(parent_df[parent_col].unique())

    orphans = child_keys - parent_keys
    status  = "OK" if len(orphans) == 0 else f"WARNING - {len(orphans)} orphan keys"

    print(f"  {label:<35} {status}")

Referential integrity checks:
  order_items.order_id                OK
  order_items.product_id              OK
  order_items.seller_id               OK
  payments.order_id                   OK
  reviews.order_id                    OK
  orders.customer_id                  OK


## STEP 3: TRANSFORM

In [9]:
# ==========================================
# 3.1 FIX DATA TYPES - DATETIME CONVERSION
# ==========================================

# Date columns identified in Step 2 as object (string) type
date_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_items": [
        "shipping_limit_date"
    ],
    "reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

# Convert each column to datetime - errors='coerce' turns unparseable values into NaT
for table_name, columns in date_columns.items():
    df = dataframes[table_name]
    for col in columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")
    print(f"  {table_name:<15} datetime columns converted: {columns}")

print("\nDatetime conversion complete.")

  orders          datetime columns converted: ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
  order_items     datetime columns converted: ['shipping_limit_date']
  reviews         datetime columns converted: ['review_creation_date', 'review_answer_timestamp']

Datetime conversion complete.


In [10]:
# ==========================================
# 3.2 HANDLE NULL VALUES
# ==========================================

# ORDERS: Null dates in delivery columns are expected (cancelled/in-transit orders)
# No action needed - NaT is the correct representation for these cases

# PRODUCTS: Fill missing category name with placeholder
dataframes["products"]["product_category_name"] = (
    dataframes["products"]["product_category_name"].fillna("unknown")
)

# PRODUCTS: Fill missing physical dimensions with column median
dim_cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
for col in dim_cols:
    median_val = dataframes["products"][col].median()
    dataframes["products"][col] = dataframes["products"][col].fillna(median_val)

# REVIEWS: Null titles and messages are expected (users who only gave a star rating)
# No action needed - nulls are valid business data here

# Verification
print("Null handling complete. Remaining nulls in affected tables:")
print()

for name in ["orders", "products", "reviews"]:
    df = dataframes[name]
    null_counts = df.isnull().sum()
    null_counts = null_counts[null_counts > 0]
    print(f"  TABLE: {name.upper()}")
    if null_counts.empty:
        print("    No null values remaining.")
    else:
        for col, count in null_counts.items():
            print(f"    {col:<40} {count:>6} nulls")
    print()

Null handling complete. Remaining nulls in affected tables:

  TABLE: ORDERS
    order_approved_at                           160 nulls
    order_delivered_carrier_date               1783 nulls
    order_delivered_customer_date              2965 nulls

  TABLE: PRODUCTS
    product_name_lenght                         610 nulls
    product_description_lenght                  610 nulls
    product_photos_qty                          610 nulls

  TABLE: REVIEWS
    review_comment_title                      87656 nulls
    review_comment_message                    58247 nulls



In [11]:
# ==========================================
# 3.3 TRANSLATE PRODUCT CATEGORIES
# ==========================================

# The products table has category names in Portuguese
# The translations table maps Portuguese names to English
# Merge to add English category column to products

dataframes["products"] = dataframes["products"].merge(
    dataframes["translations"],
    on="product_category_name",
    how="left"
)

# For products with category 'unknown' or no translation found, fill with 'unknown'
dataframes["products"]["product_category_name_english"] = (
    dataframes["products"]["product_category_name_english"].fillna("unknown")
)

# Verify result
total = len(dataframes["products"])
translated = (dataframes["products"]["product_category_name_english"] != "unknown").sum()
unknown = total - translated

print("Category translation complete.")
print(f"  Total products   : {total:,}")
print(f"  Translated       : {translated:,} ({translated/total*100:.1f}%)")
print(f"  Unknown/no match : {unknown:,} ({unknown/total*100:.1f}%)")
print()
print("Sample:")
print(
    dataframes["products"][["product_category_name", "product_category_name_english"]]
    .drop_duplicates()
    .head(8)
    .to_string(index=False)
)

Category translation complete.
  Total products   : 32,951
  Translated       : 32,328 (98.1%)
  Unknown/no match : 623 (1.9%)

Sample:
product_category_name product_category_name_english
           perfumaria                     perfumery
                artes                           art
        esporte_lazer                sports_leisure
                bebes                          baby
utilidades_domesticas                    housewares
instrumentos_musicais           musical_instruments
           cool_stuff                    cool_stuff
     moveis_decoracao               furniture_decor


In [12]:
# ==========================================
# 3.4 FEATURE ENGINEERING - BUSINESS METRICS
# ==========================================

# --- ORDERS: Delivery performance metrics ---

df_orders = dataframes["orders"]

# Actual delivery time in days (from purchase to delivery)
df_orders["delivery_days_actual"] = (
    df_orders["order_delivered_customer_date"] - df_orders["order_purchase_timestamp"]
).dt.days

# Estimated delivery time in days (from purchase to estimated date)
df_orders["delivery_days_estimated"] = (
    df_orders["order_estimated_delivery_date"] - df_orders["order_purchase_timestamp"]
).dt.days

# Delivery delay in days: positive = late, negative = early, 0 = on time
df_orders["delivery_delay_days"] = (
    df_orders["delivery_days_actual"] - df_orders["delivery_days_estimated"]
)

# Binary flag: 1 if delivered on time or early, 0 if late
df_orders["delivered_on_time"] = (df_orders["delivery_delay_days"] <= 0).astype("Int64")

# Purchase month and year for time series analysis
df_orders["purchase_year"]  = df_orders["order_purchase_timestamp"].dt.year
df_orders["purchase_month"] = df_orders["order_purchase_timestamp"].dt.month

dataframes["orders"] = df_orders

# --- ORDER_ITEMS: Revenue metrics ---

df_items = dataframes["order_items"]

# Total order line value including freight
df_items["line_total"] = df_items["price"] + df_items["freight_value"]

dataframes["order_items"] = df_items

# Verification
print("Feature engineering complete.")
print()
print("New columns in ORDERS:")
new_order_cols = ["delivery_days_actual", "delivery_days_estimated",
                  "delivery_delay_days", "delivered_on_time",
                  "purchase_year", "purchase_month"]
print(dataframes["orders"][new_order_cols].describe().to_string())
print()
print("New column in ORDER_ITEMS:")
print(dataframes["order_items"]["line_total"].describe().to_string())

Feature engineering complete.

New columns in ORDERS:
       delivery_days_actual  delivery_days_estimated  delivery_delay_days  delivered_on_time  purchase_year  purchase_month
count          96476.000000             99441.000000         96476.000000            99441.0   99441.000000    99441.000000
mean              12.094086                23.403958           -11.280142           0.896693    2017.539838        6.032220
std                9.551746                 8.829562            10.193898           0.304362       0.505007        3.232999
min                0.000000                 1.000000          -146.000000                0.0    2016.000000        1.000000
25%                6.000000                18.000000           -16.000000                1.0    2017.000000        3.000000
50%               10.000000                23.000000           -12.000000                1.0    2018.000000        6.000000
75%               15.000000                28.000000            -7.000000     

In [13]:
# ==========================================
# 3.5 BUILD ANALYTICAL MASTER TABLE
# ==========================================

# Join key tables into a single analytical dataset
# This table will be used for business queries in Step 5

# Start with order_items as the base (one row per product per order)
df_master = dataframes["order_items"].copy()

# Join orders to get status, dates and delivery metrics
df_master = df_master.merge(
    dataframes["orders"][[
        "order_id", "customer_id", "order_status",
        "order_purchase_timestamp", "order_delivered_customer_date",
        "delivery_days_actual", "delivery_delay_days",
        "delivered_on_time", "purchase_year", "purchase_month"
    ]],
    on="order_id",
    how="left"
)

# Join products to get category
df_master = df_master.merge(
    dataframes["products"][[
        "product_id", "product_category_name_english"
    ]],
    on="product_id",
    how="left"
)

# Join sellers to get seller state
df_master = df_master.merge(
    dataframes["sellers"][[
        "seller_id", "seller_state"
    ]],
    on="seller_id",
    how="left"
)

# Join customers to get customer state
df_master = df_master.merge(
    dataframes["customers"][[
        "customer_id", "customer_state"
    ]],
    on="customer_id",
    how="left"
)

# Verification
print("Master table built successfully.")
print(f"  Shape            : {df_master.shape[0]:,} rows x {df_master.shape[1]} columns")
print(f"  Columns          : {df_master.columns.tolist()}")
print(f"  Null check       :")
nulls = df_master.isnull().sum()
nulls = nulls[nulls > 0]
if nulls.empty:
    print("    No null values found.")
else:
    for col, count in nulls.items():
        print(f"    {col:<45} {count:>6} nulls")

Master table built successfully.
  Shape            : 112,650 rows x 20 columns
  Columns          : ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'line_total', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_delivered_customer_date', 'delivery_days_actual', 'delivery_delay_days', 'delivered_on_time', 'purchase_year', 'purchase_month', 'product_category_name_english', 'seller_state', 'customer_state']
  Null check       :
    order_delivered_customer_date                   2454 nulls
    delivery_days_actual                            2454 nulls
    delivery_delay_days                             2454 nulls


## STEP 4: LOAD

In [14]:
# ==========================================
# 4.1 CREATE SQLITE DATABASE
# ==========================================

# Output path for the processed data and database
PROCESSED_PATH = Path("data/processed")
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

DB_PATH = PROCESSED_PATH / "olist_etl.db"

# Create SQLAlchemy engine connected to SQLite database file
engine = create_engine(f"sqlite:///{DB_PATH}")

print(f"Database engine created.")
print(f"  Location : {DB_PATH}")
print(f"  Dialect  : {engine.dialect.name}")

Database engine created.
  Location : data\processed\olist_etl.db
  Dialect  : sqlite


In [15]:
# ==========================================
# 4.2 LOAD TABLES TO DATABASE
# ==========================================

# Define which tables to load and their primary keys
tables_to_load = {
    "orders":       dataframes["orders"],
    "order_items":  dataframes["order_items"],
    "customers":    dataframes["customers"],
    "products":     dataframes["products"],
    "sellers":      dataframes["sellers"],
    "payments":     dataframes["payments"],
    "reviews":      dataframes["reviews"],
    "master":       df_master
}

# Load each table into SQLite
# if_exists='replace' drops and recreates the table on each run
# index=False avoids writing the pandas index as a column
print("Loading tables into SQLite database...")
print()

for table_name, df in tables_to_load.items():
    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="replace",
        index=False
    )
    print(f"  {table_name:<15} loaded   {len(df):>7,} rows   {len(df.columns)} columns")

print()
print("All tables loaded successfully.")

Loading tables into SQLite database...

  orders          loaded    99,441 rows   14 columns
  order_items     loaded   112,650 rows   8 columns
  customers       loaded    99,441 rows   5 columns
  products        loaded    32,951 rows   10 columns
  sellers         loaded     3,095 rows   4 columns
  payments        loaded   103,886 rows   5 columns
  reviews         loaded    99,224 rows   7 columns
  master          loaded   112,650 rows   20 columns

All tables loaded successfully.


In [16]:
# ==========================================
# 4.3 LOAD VERIFICATION
# ==========================================

# Query the SQLite database to confirm all tables were loaded correctly
# This validates the Load step independently from the pandas DataFrames

print("Verifying database contents...")
print()

with engine.connect() as conn:

    # List all tables in the database
    result = conn.execute(text("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"))
    db_tables = [row[0] for row in result]
    print(f"  Tables in database : {db_tables}")
    print()

    # Row count per table directly from SQLite
    print("  Row counts from SQLite:")
    for table in db_tables:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table}"))
        count = result.fetchone()[0]
        print(f"    {table:<15} {count:>7,} rows")

print()
print("Load verification complete.")

Verifying database contents...

  Tables in database : ['customers', 'master', 'order_items', 'orders', 'payments', 'products', 'reviews', 'sellers']

  Row counts from SQLite:
    customers        99,441 rows
    master          112,650 rows
    order_items     112,650 rows
    orders           99,441 rows
    payments        103,886 rows
    products         32,951 rows
    reviews          99,224 rows
    sellers           3,095 rows

Load verification complete.


## STEP 5: VALIDATE & QUERY

In [17]:
# ==========================================
# 5.1 BUSINESS QUERIES
# ==========================================

with engine.connect() as conn:

    # --- QUERY 1: On-time delivery rate by year ---
    print("QUERY 1: On-time delivery rate by year")
    print("-" * 45)
    q1 = text("""
        SELECT
            purchase_year,
            COUNT(*)                                        AS total_orders,
            SUM(delivered_on_time)                         AS on_time,
            ROUND(AVG(delivered_on_time) * 100, 2)         AS on_time_pct
        FROM orders
        WHERE order_status = 'delivered'
        GROUP BY purchase_year
        ORDER BY purchase_year
    """)
    print(pd.read_sql(q1, conn).to_string(index=False))
    print()

    # --- QUERY 2: Top 10 product categories by revenue ---
    print("QUERY 2: Top 10 product categories by revenue")
    print("-" * 45)
    q2 = text("""
        SELECT
            product_category_name_english           AS category,
            COUNT(DISTINCT order_id)                AS total_orders,
            ROUND(SUM(line_total), 2)               AS total_revenue,
            ROUND(AVG(price), 2)                    AS avg_price
        FROM master
        WHERE order_status = 'delivered'
        GROUP BY category
        ORDER BY total_revenue DESC
        LIMIT 10
    """)
    print(pd.read_sql(q2, conn).to_string(index=False))
    print()

    # --- QUERY 3: Average delivery delay by seller state ---
    print("QUERY 3: Average delivery delay by seller state (top 10 worst)")
    print("-" * 45)
    q3 = text("""
        SELECT
            seller_state,
            COUNT(DISTINCT order_id)                AS total_orders,
            ROUND(AVG(delivery_delay_days), 1)      AS avg_delay_days
        FROM master
        WHERE order_status = 'delivered'
          AND delivery_delay_days IS NOT NULL
        GROUP BY seller_state
        ORDER BY avg_delay_days DESC
        LIMIT 10
    """)
    print(pd.read_sql(q3, conn).to_string(index=False))
    print()

    # --- QUERY 4: Monthly order volume trend ---
    print("QUERY 4: Monthly order volume (2017-2018)")
    print("-" * 45)
    q4 = text("""
        SELECT
            purchase_year   AS year,
            purchase_month  AS month,
            COUNT(*)        AS total_orders,
            ROUND(SUM(payment_value), 2) AS total_revenue
        FROM orders
        JOIN payments USING (order_id)
        WHERE purchase_year IN (2017, 2018)
          AND order_status = 'delivered'
        GROUP BY purchase_year, purchase_month
        ORDER BY purchase_year, purchase_month
    """)
    print(pd.read_sql(q4, conn).to_string(index=False))

QUERY 1: On-time delivery rate by year
---------------------------------------------
 purchase_year  total_orders  on_time  on_time_pct
          2016           267      264        98.88
          2017         43428    40744        93.82
          2018         52783    48155        91.23

QUERY 2: Top 10 product categories by revenue
---------------------------------------------
             category  total_orders  total_revenue  avg_price
        health_beauty          8647     1412089.53     130.28
        watches_gifts          5495     1264333.12     199.04
       bed_bath_table          9272     1225209.26      93.44
       sports_leisure          7530     1118256.91     113.25
computers_accessories          6530     1032723.77     116.26
      furniture_decor          6307      880329.92      87.25
           housewares          5743      758392.25      90.60
           cool_stuff          3559      691680.89     164.12
                 auto          3810      669454.75     139.8

## STEP 6: EXPORT & SUMMARY

In [18]:
# ==========================================
# 6.1 EXPORT MASTER TABLE
# ==========================================

# Export the master analytical table to CSV for external use
MASTER_CSV = PROCESSED_PATH / "olist_master.csv"
df_master.to_csv(MASTER_CSV, index=False)

print(f"Master table exported.")
print(f"  File     : {MASTER_CSV}")
print(f"  Rows     : {len(df_master):,}")
print(f"  Columns  : {len(df_master.columns)}")
print(f"  Size     : {MASTER_CSV.stat().st_size / 1024**2:.2f} MB")

Master table exported.
  File     : data\processed\olist_master.csv
  Rows     : 112,650
  Columns  : 20
  Size     : 28.32 MB


In [19]:
# ==========================================
# 6.2 PIPELINE SUMMARY
# ==========================================

print("=" * 60)
print("ETL PIPELINE SUMMARY")
print("=" * 60)
print()
print("SOURCE")
print(f"  Dataset          : Brazilian E-Commerce (Olist)")
print(f"  Files extracted  : 9 CSV files")
print(f"  Total source rows: {sum(len(df) for df in dataframes.values()):,}")
print()
print("TRANSFORM")
print(f"  Datetime columns converted : 8")
print(f"  Null strategies applied    : 3 (drop, fill median, fill placeholder)")
print(f"  Categories translated      : Portuguese to English (98.1%)")
print(f"  New features engineered    : 7")
print()
print("LOAD")
print(f"  Database         : SQLite (olist_etl.db)")
print(f"  Tables loaded    : 8")
print(f"  Master table     : {len(df_master):,} rows x {len(df_master.columns)} columns")
print(f"  Export           : olist_master.csv")
print()
print("KEY BUSINESS FINDINGS")
print(f"  On-time delivery rate (2018)  : 91.2%")
print(f"  Top revenue category          : Health & Beauty ($1.4M)")
print(f"  Peak month                    : November 2017 (Black Friday)")
print(f"  Most delayed region           : AM - Amazonas (+9.3 days avg)")
print()
print("=" * 60)
print("Pipeline completed successfully.")
print("=" * 60)

ETL PIPELINE SUMMARY

SOURCE
  Dataset          : Brazilian E-Commerce (Olist)
  Files extracted  : 9 CSV files
  Total source rows: 1,550,922

TRANSFORM
  Datetime columns converted : 8
  Null strategies applied    : 3 (drop, fill median, fill placeholder)
  Categories translated      : Portuguese to English (98.1%)
  New features engineered    : 7

LOAD
  Database         : SQLite (olist_etl.db)
  Tables loaded    : 8
  Master table     : 112,650 rows x 20 columns
  Export           : olist_master.csv

KEY BUSINESS FINDINGS
  On-time delivery rate (2018)  : 91.2%
  Top revenue category          : Health & Beauty ($1.4M)
  Peak month                    : November 2017 (Black Friday)
  Most delayed region           : AM - Amazonas (+9.3 days avg)

Pipeline completed successfully.
